#  Preprocessing, Feature Engineering & MLflow

**Tujuan:** ubah data mentah jadi format siap-model **tanpa data leakage**, sambil membangun langkah-langkah yang bisa dipakai ulang di FastAPI (Hari 5).

Recap insight Hari 1 (EDA) yang jadi dasar feature engineering:
- `Contract` Month-to-month & `tenure` pendek = paling rawan churn
- Punya banyak add-on (`OnlineSecurity` dll) menahan churn
- Kelompok `No internet service` churn-nya sangat rendah (7.4%)

**Prinsip kunci:** logika preprocessing tidak ditulis ad-hoc di notebook, tapi di modul `src/preprocessing.py` supaya kode yang **sama persis** dipakai saat training maupun saat prediksi data baru di API.

## 0. Setup & Import

In [1]:
import sys
sys.path.append('..')  # supaya bisa import modul dari folder src/

import pandas as pd
import joblib
from sklearn.model_selection import train_test_split

from src.preprocessing import (
    clean_raw, add_features, split_columns,
    build_preprocessor, prepare_xy,
)

df_raw = pd.read_csv('../data/raw/Telco-Customer-Churn.csv')
print('Shape data mentah:', df_raw.shape)

Shape data mentah: (7043, 21)


## 1. Cleaning

`clean_raw()` memperbaiki `TotalCharges`: 11 baris berisi spasi `' '` karena `tenure = 0` (pelanggan baru, belum pernah ditagih). Diubah ke numeric lalu diisi 0.

> Catatan: di project nyata, semua langkah cleaning yang kita temukan saat EDA dipindah ke fungsi reusable seperti ini, bukan diketik ulang tiap notebook.

In [2]:
df_clean = clean_raw(df_raw)
print('Tipe TotalCharges sekarang:', df_clean['TotalCharges'].dtype)
print('Missing value total       :', df_clean.isnull().sum().sum())

Tipe TotalCharges sekarang: float64
Missing value total       : 0


## 2. Feature Engineering

`add_features()` membuat 4 fitur baru — semuanya **row-wise** (cuma melihat baris itu sendiri), jadi **aman** dijalankan sebelum train/test split:

| Fitur baru | Definisi | Alasan (dari EDA) |
|---|---|---|
| `num_addons` | jumlah add-on aktif (0–6) | makin sedikit add-on, makin tinggi churn |
| `is_new_customer` | 1 jika `tenure` ≤ 12 | churner rata-rata tenure 18 bln |
| `has_internet` | 1 jika langganan internet | kelompok 'No internet' churn rendah |
| `tenure_group` | binning tenure (4 fase) | memberi model sinyal fase masa langganan |

In [3]:
df_feat = add_features(df_clean)

cols_baru = ['tenure', 'tenure_group', 'num_addons', 'is_new_customer', 'has_internet']
df_feat[cols_baru].head()

,tenure,tenure_group,num_addons,is_new_customer,has_internet
0,1,0-12,1,1,1
1,34,25-48,2,0,1
2,2,0-12,2,1,1
3,45,25-48,3,0,1
4,2,0-12,0,1,1


In [4]:
# Validasi: apakah fitur baru benar-benar membedakan churn?
for col in ['tenure_group', 'num_addons']:
    rate = df_feat.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).round(1)
    print(f'Churn rate per {col}:')
    print(rate.to_string())
    print('-' * 30)

Churn rate per tenure_group:
tenure_group
0-12     47.4
13-24    28.7
25-48    20.4
49-72     9.5
------------------------------
Churn rate per num_addons:
num_addons
0    21.4
1    45.8
2    35.8
3    27.4
4    22.3
5    12.4
6     5.3
------------------------------


### 📌 Insight validasi fitur baru

- **`tenure_group`** sangat diskriminatif & monotonik: `0-12` bln = **47.4%** churn → `13-24` = 28.7% → `25-48` = 20.4% → `49-72` = **9.5%**. Makin lama langganan, makin loyal.
- **`num_addons`** menahan churn mulai dari 1 add-on: 1 = **45.8%** → 3 = 27.4% → 6 = **5.3%**. 
  - ⚠️ Kategori `0` justru rendah (21.4%) karena **tercampur** pelanggan tanpa internet (yang otomatis 0 add-on & jarang churn). Inilah gunanya `has_internet` — memisahkan dua kelompok yang berbeda makna ini.

Kesimpulan: feature engineering kita menambah sinyal yang jelas, bukan sekadar kolom kosong.

## 3. Train/Test Split (stratified)

`prepare_xy()` menjalankan clean + feature engineering lalu memisah `X` dan `y` (1 = churn).

Kita split **sebelum** fit preprocessing. `stratify=y` menjaga proporsi churn (26.5%) tetap sama di train & test — penting untuk dataset imbalanced supaya evaluasi adil.

In [5]:
X, y = prepare_xy(df_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print('Train:', X_train.shape, '| churn rate:', round(y_train.mean() * 100, 2), '%')
print('Test :', X_test.shape, '| churn rate:', round(y_test.mean() * 100, 2), '%')

Train: (5634, 24) | churn rate: 26.54 %
Test : (1409, 24) | churn rate: 26.54 %


## 4. Bangun Preprocessing Pipeline (ColumnTransformer)

Inilah inti gaya production. Satu objek `ColumnTransformer` menangani:
- **numeric** (`tenure`, `MonthlyCharges`, `TotalCharges`, `num_addons`) → impute median + `StandardScaler`
- **categorical** (16 kolom) → `OneHotEncoder(handle_unknown='ignore')`
- **passthrough** (`SeniorCitizen`, `is_new_customer`, `has_internet`) → sudah 0/1, diteruskan apa adanya

### ❗ Kunci anti data-leakage
Kita `.fit()` preprocessor **HANYA pada data train**, lalu `.transform()` data test. Kenapa? Karena scaler menghitung mean/std dan encoder mempelajari daftar kategori — kalau ini ikut "mengintip" data test, model jadi seolah tahu masa depan, dan skor evaluasi jadi palsu (terlalu optimis).

In [6]:
numeric_cols, passthrough_cols, categorical_cols = split_columns(X_train)
print('Numeric    :', numeric_cols)
print('Passthrough:', passthrough_cols)
print('Categorical:', len(categorical_cols), 'kolom')

preprocessor = build_preprocessor(numeric_cols, passthrough_cols, categorical_cols)

# FIT hanya di train, lalu transform keduanya
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print('\nSebelum  :', X_train.shape[1], 'kolom mentah')
print('Sesudah  :', X_train_prep.shape[1], 'fitur (hasil one-hot encoding)')
print('Train prep:', X_train_prep.shape, '| Test prep:', X_test_prep.shape)

Numeric    : ['tenure', 'MonthlyCharges', 'TotalCharges', 'num_addons']
Passthrough: ['SeniorCitizen', 'is_new_customer', 'has_internet']
Categorical: 16 kolom

Sebelum  : 24 kolom mentah
Sesudah  : 52 fitur (hasil one-hot encoding)
Train prep: (5634, 52) | Test prep: (1409, 52)


### 📌 Insight
- 24 kolom mentah → **52 fitur** setelah one-hot encoding.
- `handle_unknown='ignore'` membuat preprocessor **tahan banting**: jika nanti di API datang nilai kategori yang belum pernah dilihat, ia tidak error — kolom one-hot-nya cukup jadi 0.
- Test di-`transform` pakai parameter yang dipelajari dari **train saja** → evaluasi nanti jujur.

## 5. Simpan Preprocessor sebagai Artifact

Preprocessor yang sudah di-`fit` kita simpan ke file `.joblib`. FastAPI nanti tinggal `joblib.load()` objek ini untuk memproses request pelanggan baru — **persis** seperti saat training. Tanpa ini, kita harus menulis ulang logika preprocessing di API (rawan beda & bug).

In [7]:
import os
os.makedirs('../models', exist_ok=True)

joblib.dump(preprocessor, '../models/preprocessor.joblib')
print('Preprocessor tersimpan -> models/preprocessor.joblib')

# (opsional) simpan juga data hasil split untuk dipakai di notebook modeling Hari 3
joblib.dump((X_train, X_test, y_train, y_test), '../models/split_data.joblib')
print('Data split tersimpan   -> models/split_data.joblib')

Preprocessor tersimpan -> models/preprocessor.joblib
Data split tersimpan   -> models/split_data.joblib


## 6. MLflow — Pengenalan

**Analogi:** MLflow itu **buku catatan + gudang model**. Dia tidak melatih atau men-tuning apa pun — dia **MEREKAM** setiap eksperimen: parameter apa yang dipakai, metrik berapa, dan menyimpan artifact (model, preprocessor).

Kenapa penting (dan ditanya saat interview tech besar):
- Bisa **membandingkan** puluhan eksperimen tanpa pusing ("model mana yang recall-nya paling tinggi?")
- Bisa **rollback** ke versi model lama kapan pun
- **Reproducible**: tercatat persis konfigurasi yang menghasilkan suatu skor

Istilah inti:
- **Experiment** = folder besar (mis. "telco-churn")
- **Run** = satu kali percobaan di dalamnya
- **Param** = input yang kamu pilih (test_size, dll) · **Metric** = output (akurasi) · **Artifact** = file (preprocessor/model)

Hari 2 kita cuma **cicipi**: catat 1 run untuk langkah preprocessing. Hari 3 baru dipakai penuh untuk membandingkan banyak model.

In [8]:
import mlflow
from pathlib import Path

# Kumpulkan SEMUA catatan MLflow di root project (1 folder di atas notebooks/),
# bukan di dalam notebooks/, supaya rapi dan konsisten dari mana pun dijalankan.
ROOT = Path.cwd().parent
mlflow.set_tracking_uri(f'sqlite:///{(ROOT / "mlflow.db").as_posix()}')

# Buat experiment dengan lokasi artifact eksplisit di root/mlartifacts (sekali saja)
EXP = 'telco-churn'
if mlflow.get_experiment_by_name(EXP) is None:
    mlflow.create_experiment(EXP, artifact_location=(ROOT / 'mlartifacts').as_uri())
mlflow.set_experiment(EXP)

with mlflow.start_run(run_name='preprocessing-hari2'):
    # PARAM = pilihan kita
    mlflow.log_param('test_size', 0.2)
    mlflow.log_param('random_state', 42)
    mlflow.log_param('n_engineered_features', 4)

    # METRIC = hasil terukur
    mlflow.log_metric('n_features_raw', X_train.shape[1])
    mlflow.log_metric('n_features_after_prep', X_train_prep.shape[1])
    mlflow.log_metric('train_churn_rate', float(y_train.mean()))

    # ARTIFACT = file hasil kerja
    mlflow.log_artifact('../models/preprocessor.joblib')

    print('Run preprocessing tercatat di MLflow OK')

2026/06/22 02:22:56 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/06/22 02:22:56 INFO mlflow.store.db.utils: Updating database tables


Run preprocessing tercatat di MLflow OK


### 👀 Lihat hasilnya di MLflow UI

Jalankan di terminal (dari folder root project):

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

lalu buka http://localhost:5000 — kamu akan lihat experiment `telco-churn` dengan run `preprocessing-hari2`, lengkap dengan params, metrics, dan artifact `preprocessor.joblib`.

---
## ✅ Selesai Hari 2 — yang sudah kita punya
- `src/preprocessing.py` — logika reusable (dipakai lagi di FastAPI Hari 5)
- `models/preprocessor.joblib` — preprocessor ter-fit
- `models/split_data.joblib` — data train/test siap pakai
- `mlflow.db` — 1 run MLflow pertama tercatat

## ➡️ Next — Hari 3: Modeling & Experiment Tracking
- Latih beberapa model (LogisticRegression, RandomForest, XGBoost)
- Tangani imbalance (`class_weight='balanced'`)
- **Log setiap model ke MLflow** lalu bandingkan (fokus metrik **recall/F1**, bukan akurasi, karena tujuan bisnis = menangkap calon churner)